In [8]:
from matplotlib import pyplot as plt
import plotly.graph_objects as go
import yfinance as yf
import pandas as pd
import numpy as np
import os
import sys
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from plotly.subplots import make_subplots

In [9]:
ES_TICKER = "GBPUSD=X"
NQ_TICKER = "EURUSD=X"

es_df = yf.download(ES_TICKER, period="max", interval="1h")
nq_df = yf.download(NQ_TICKER, period="max", interval="1h")

es_df.columns.names = [None, None]
nq_df.columns.names = [None, None]

es_df.columns = es_df.columns.get_level_values(0)
nq_df.columns = nq_df.columns.get_level_values(0)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [10]:
df = (
    es_df[["Close"]]
    .rename(columns={"Close": ES_TICKER})
    .join(
        nq_df[["Close"]].rename(columns={"Close": NQ_TICKER}),
        how="inner"
    )
)

df

,GBPUSD=X,EURUSD=X
Datetime,,
2024-07-24 18:00:00+01:00,1.291856,1.084952
2024-07-24 19:00:00+01:00,1.290489,1.084011
2024-07-24 20:00:00+01:00,1.290422,1.084246
2024-07-24 21:00:00+01:00,1.290722,1.084363
2024-07-24 22:00:00+01:00,1.290889,1.084363
...,...,...
2026-07-24 14:00:00+01:00,1.332676,1.137915
2026-07-24 15:00:00+01:00,1.332836,1.138304
2026-07-24 16:00:00+01:00,1.334116,1.138693


## Plot of both tickers

In [11]:
# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Left y-axis
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df[ES_TICKER],
        mode="lines",
        name=ES_TICKER
    ),
    secondary_y=False
)

# Right y-axis
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df[NQ_TICKER],
        mode="lines",
        name=NQ_TICKER
    ),
    secondary_y=True
)

# Axis titles
fig.update_xaxes(title_text="Time")
fig.update_yaxes(title_text="Price", secondary_y=False)
fig.update_yaxes(title_text="Price", secondary_y=True)

fig.update_layout(
    title=f"{NQ_TICKER} vs {ES_TICKER}",
    xaxis_title="Date",
    yaxis_title="price",
    template="plotly_dark",
    height=600,
    width=1500
)

fig.show()

## Correlation

In [12]:
df["Ratio"] = nq_df["Close"]/es_df["Close"]

MA_WINDOW = 200
df[f'ratio_ma({MA_WINDOW})'] = df['Ratio'].rolling(window=MA_WINDOW, center=False).mean()

df.tail()

,GBPUSD=X,EURUSD=X,Ratio,ratio_ma(200)
Datetime,,,,
2026-07-24 14:00:00+01:00,1.332676,1.137915,0.853858,0.851581
2026-07-24 15:00:00+01:00,1.332836,1.138304,0.854047,0.851588
2026-07-24 16:00:00+01:00,1.334116,1.138693,0.853519,0.851592
2026-07-24 17:00:00+01:00,1.333422,1.138174,0.853574,0.851593
2026-07-24 18:00:00+01:00,1.332250,1.137139,0.853548,0.851598


In [13]:
import plotly.express as px


fig = go.Figure()

# First series
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Ratio'],
    mode='lines',
    name=f"{NQ_TICKER} / {ES_TICKER} Ratio"
))

# Second series
fig.add_trace(go.Scatter(
    x=df.index,
    y=df[f'ratio_ma({MA_WINDOW})'],
    mode='lines',
    name=f"Ratio Moving Average({MA_WINDOW})"
))

fig.update_layout(
    title=f"{NQ_TICKER} / {ES_TICKER} Ratio and {MA_WINDOW}-period Moving Average",
    xaxis_title="Date",
    yaxis_title="Ratio",
    template="plotly_dark",
    height=600,
    width=1500
)

fig.show()